# TNBC miRNA–mRNA GAT study, phase 3 retry (kaggle-phase3r): redesigned planted control and variant selection

**Post-hoc, added after the frozen study.** This is the single retry of part A that Heram approved after
phase 3a (`kaggle-phase3`, commit `5a5bec9`) failed its planted control: joint_nocross read the first planted
signal as well as joint_mirdb, so the control did not require edges. Only the control changed. The four
edge-forcing variants and the pass rule are the same as in phase 3a. Settings and rules: `kaggle/phase3_config.py`.

| Step | What | GAT fits |
|---|---|---|
| Design gate | Graph-free models (elastic net, XGBoost per modality and joint, frozen joint_nocross) must be near chance (mean inner-validation balanced accuracy ≤ 0.43) at δ = 1 on repeat 0. **The notebook stops here if the gate fails.** | 5 (joint_nocross only) |
| A | Redesigned planted control: frozen joint_mirdb / joint_nocross / joint_rewired + routed baselines, δ ∈ {0, 0.5, 0.75, 1}, repeat 0, 5 outer folds × 3 seeds | 180 |
| B | Variant selection (xfirst, gated, selfdrop 0.25 / 0.5) on the redesigned planted task, inner-validation split only, then real-data inner-validation loss. Writes `selected_variant.json`. | 280 (240 if phase 3a's output is attached) |

The planted signal: each example gets four latent values, one per miRDB hub set (targets of four hub miRNAs in
mRNA examples, regulators of four hub genes in miRNA examples), added to every member of that set. The class fixes
only the signs of the two within-pair products, so no feature, set mean or co-expression strength differs between
classes; the class is readable only by knowing which features form each miRDB set.

Part C finished in phase 3a and is not rerun.

**Settings:** Internet **on**, accelerator **None (CPU)**. **Attach phase 3a's output** (*Add Input → Your Work*)
so the 40 real-data selection fits, which this retry does not change, are reused.

**Smoke test first** (`SMOKE_TEST = True`), then `SMOKE_TEST = False` and *Save Version → Save & Run All*.

**After it finishes:** download `phase3_output/selected_variant.json`. If `"selected": true`, commit it as
`kaggle/phase3_frozen/selected_variant.json` on `kaggle-phase3r` and run `phase3b.ipynb`. If false, no variant passed and
phase 3b is not run.

In [ ]:
import os, subprocess, time

SMOKE_TEST = True          # True: one fold, one seed per arm type and a runtime estimate. False: everything.
WORKERS = os.cpu_count()   # parallel fits; each fit is single-threaded (frozen setting)
TIME_BUDGET_H = 11.0       # Kaggle sessions stop at 12 h
1.0_H = 1.0        # kept for baselines, analysis and the zip; no new GAT fit starts after budget - reserve

BRANCH = 'kaggle-phase3r'
REPO_URL = 'https://github.com/heramn/TNBC-miRNA-mRNA-GNN.git'
SRC = '/tmp/TNBC-miRNA-mRNA-GNN'
PY = '/tmp/p3env/bin/python'
WORK = '/kaggle/working'
T0 = time.time()
DEADLINE = T0 + (TIME_BUDGET_H - 1.0_H) * 3600
os.environ.update(STUDY_RUNS=f'{WORK}/phase3_runs', PHASE3_OUT=f'{WORK}/phase3_output', STUDY_CACHE='/tmp/study_cache',
                  OMP_NUM_THREADS='1', OPENBLAS_NUM_THREADS='1', MKL_NUM_THREADS='1')


def sh(cmd, check=True):
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    if p.wait() and check:
        raise RuntimeError(f'exit {p.returncode}: {cmd}')
    return p.returncode

SM = '--smoke' if SMOKE_TEST else ''
print('workers:', WORKERS, '| smoke test:', SMOKE_TEST)

## 1. Clone the repository at `kaggle-phase3r`

In [ ]:
sh(f'rm -rf {SRC} && git clone -q -b {BRANCH} {REPO_URL} {SRC} && git -C {SRC} log -1 --format="%h %s"')

## 2. Python environment

The same isolated Python 3.12 venv as phase 2, with phase 1's pins: torch **2.15.0+cpu**, torch_geometric **2.8.1**,
scikit-learn **1.7.2** (1.8 changes the grouped splits). Kaggle's preinstalled CUDA torch is left alone.

In [ ]:
sh('pip install -q uv')
sh('uv venv -q --python 3.12 /tmp/p3env')
sh(f'uv pip install -q --python {PY} numpy==2.5.5 pandas==3.1.1 scipy==1.18.1 scikit-learn==1.7.2 statsmodels==0.15.0 '
   'matplotlib==3.11.2 xgboost==3.4.1 pydeseq2==0.6.0 anndata==0.13.4 torch_geometric==2.8.1 pytest')
sh(f'uv pip install -q --python {PY} torch==2.15.0 --index-url https://download.pytorch.org/whl/cpu')
sh(f'{PY} -c "import sys, torch, torch_geometric, sklearn, numpy; print(sys.version.split()[0], torch.__version__, '
   'torch_geometric.__version__, sklearn.__version__, numpy.__version__)"')

## 3. Data, checkpoint restore, frozen-file tests, plan, planted-signal check

`check-plant` (no model fit) prints, per fold and strength, the hub set sizes, the largest single-feature class
signal in mean (η²) and spread (class variance ratio), and the balanced accuracy of the oracle, a logistic model
given the two true set-mean products (chance 0.33).

In [ ]:
os.chdir(SRC)
sh(f'{PY} kaggle/phase3.py data')
sh(f'{PY} kaggle/phase3.py restore')
sh(f'{PY} -m pytest -q -p no:cacheprovider study/tests')
sh(f'{PY} kaggle/phase3.py plan')
sh(f'{PY} kaggle/phase3.py check-plant')

## 4. Design gate (before any edge-using GAT fit)

Five joint_nocross fits (configuration 0, seed 0) on the inner-validation split at δ = 1, then the tabular
graph-free and edge-aware models. `gate` writes `phase3_output/tables/p3r_gate.csv` and **raises (stopping the
notebook) if any graph-free model is above 0.43 mean balanced accuracy**.

In [ ]:
sh(f'{PY} kaggle/phase3.py run --part gate {SM} --workers {WORKERS} --deadline {DEADLINE:.0f}')
if SMOKE_TEST:
    print('smoke test: gate verdict needs all 5 folds; skipped')
else:
    sh(f'{PY} kaggle/phase3.py gate')

## 5. Part A: redesigned planted-signal positive control

In [ ]:
sh(f'{PY} kaggle/phase3.py run --part A {SM} --workers {WORKERS} --deadline {DEADLINE:.0f}')
sh(f'{PY} kaggle/phase3.py baselines --part A {SM} --workers {WORKERS}')

## 6. Part B: edge-forcing variant selection (inner validation only, same rule as phase 3a)

Four candidates × {miRDB, no-cross, rewired} edges × δ ∈ {0, 1} on the redesigned planted task, scored on the
inner-validation split, plus each candidate with miRDB edges on the real labels (inner-validation loss; reused from
phase 3a when its output is attached). `select` applies the rule in `phase3_config.py` and writes
`phase3_output/selected_variant.json`.

In [ ]:
sh(f'{PY} kaggle/phase3.py run --part B {SM} --workers {WORKERS} --deadline {DEADLINE:.0f}')
if SMOKE_TEST:
    sh(f'{PY} kaggle/phase3.py estimate --workers {WORKERS}')
else:
    rc = sh(f'{PY} kaggle/phase3.py select', check=False)
    print('selection', 'written' if rc == 0 else f'not complete (exit {rc}); attach this output and rerun')

## 7. Analysis and RESULTS rows (full mode)

In [ ]:
if SMOKE_TEST:
    print('smoke test: analysis skipped')
else:
    rc = sh(f'{PY} kaggle/phase3.py analyze --part A', check=False)
    print('part A analysis', 'done' if rc == 0 else f'failed (exit {rc})')
    sh(f'{PY} kaggle/phase3.py report', check=False)
    sh(f'{PY} kaggle/phase3.py plan')
    sel = f'{WORK}/phase3_output/selected_variant.json'
    if os.path.exists(sel):
        print(open(sel).read())

## 8. Zip the outputs

`phase3_runs/` is the checkpoint (every per-run CSV); `phase3_output/` has the gate and result tables, `selected_variant.json`, the fit log and `RESULTS_phase3_rows.md`.

In [ ]:
import shutil, zipfile
os.chdir(WORK)
z = shutil.make_archive(f'{WORK}/tnbc_phase3r', 'zip', WORK, '.')
with zipfile.ZipFile(z) as zf:
    names = [n for n in zf.namelist() if n.startswith('phase3_runs/') and n.endswith('.csv')]
print(z, f'{os.path.getsize(z) / 1e6:.1f} MB,', len(names), 'per-run CSVs')
rows = f'{WORK}/phase3_output/RESULTS_phase3_rows.md'
if os.path.exists(rows):
    print(open(rows).read())
print(f'total time {(time.time() - T0) / 3600:.2f} h')